In [ ]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from sklearn.utils import resample
from sklearn.utils.class_weight import compute_class_weight
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer, 
    DataCollatorWithPadding
)
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import evaluate

In [ ]:
# Data cleaning function
def clean_data(df):
    df_clean = df.copy()
    df_clean = df_clean[["reviews.text", "reviews.title", "reviews.rating"]]
    df_clean = df_clean.dropna(subset=["reviews.text"])
    df_clean = df_clean.dropna(subset=["reviews.title"])

    label_map = {
        5: 2,
        4: 2,
        3: 1,
        2: 0,
        1: 0
    }
    df_clean["label"] = df_clean["reviews.rating"].map(label_map)
    df_clean = df_clean.dropna(subset=["label"])
    df_clean["label"] = df_clean["label"].astype(int)
    df_clean = df_clean.drop_duplicates(subset=["reviews.text"], keep="first")
    df_clean["full_text"] = df_clean["reviews.title"] + " " + df_clean["reviews.text"]

    return df_clean[["full_text", "label"]]

In [ ]:
# Load and clean full dataset
data = pd.read_csv("..data/1429_1.csv")
data_clean = clean_data(data)

In [ ]:
# Oversampling 
df_pos = data_clean[data_clean["label"] == 2]
df_neu = data_clean[data_clean["label"] == 1]
df_neg = data_clean[data_clean["label"] == 0]

df_neu_upsampled = resample(df_neu, replace=True, n_samples=5000, random_state=42)
df_neg_upsampled = resample(df_neg, replace=True, n_samples=5000, random_state=42)

df_final = pd.concat([df_pos, df_neu_upsampled, df_neg_upsampled])
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
# Class weights for imbalance
classes = np.unique(data_clean["label"])
weights = compute_class_weight(
    class_weight="balanced", 
    classes=classes, 
    y=data_clean["label"]
)
class_weights = torch.tensor(weights, dtype=torch.float)

In [ ]:
# Prepare dataset 
raw_dataset = Dataset.from_dict(df_final)
split_data = raw_dataset.train_test_split(test_size=0.2, seed=42)

In [ ]:
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess_function(batch):
    return tokenizer(batch["full_text"], truncation=True, padding="max_length", max_length=128)

tokenized_datasets = split_data.map(preprocess_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
# Metrics and custom weighted trainer
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # Move weights to the correct device (GPU/CPU)
        weights_tensor = class_weights.to(self.args.device)
        loss_fct = nn.CrossEntropyLoss(weight=weights_tensor)
        
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [ ]:
# Initialize model 
id2label = {0: "negative", 1: "neutral", 2: "positive"}
label2id = {"negative": 0, "neutral": 1, "positive": 2}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=3, id2label=id2label, label2id=label2id
)

# Training arguments 
training_args = TrainingArguments(
    fp16=True,
    output_dir="review_classifier_weighted",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,
)

# Run trainer
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer.train()

In [ ]:
# Custom thresholding 
def get_custom_predictions(logits, threshold_shift=0.1):
    probs = torch.nn.functional.softmax(torch.tensor(logits), dim=-1).numpy()

    probs[:, 0] += threshold_shift  # Give Negative a 10% boost
    probs[:, 1] += threshold_shift  # Give Neutral a 10% boost
    
    return np.argmax(probs, axis=1)

raw_preds = trainer.predict(tokenized_datasets["test"])
custom_preds = get_custom_predictions(raw_preds.predictions)

In [ ]:
test_results = trainer.predict(tokenized_datasets["test"])

# The predictions are in logit format; use argmax to get the predicted class indices
y_pred = np.argmax(test_results.predictions, axis=-1)
y_true = test_results.label_ids

# Define the labels for visualization
target_names = ["negative", "neutral", "positive"]

# Create confusion matrix 
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, 
    annot=True, 
    fmt="d", 
    cmap="Blues", 
    xticklabels=target_names, 
    yticklabels=target_names
)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# Metrics table 
report = classification_report(
    y_true, 
    y_pred, 
    target_names=target_names, 
    output_dict=True
)

# Convert to DataFrame and format for readability
metrics_df = pd.DataFrame(report).transpose()

# Display the table
print("Classification Metrics Table:")
print(metrics_df)